# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','onnxsim':'onnxsim','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import collections, shutil
import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
from onnxsim import simplify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 74.9 MB/s eta 0:00:00


In [4]:
TASK_ID='task074'; CH=10; H=W=30


ROOT=Path(COMPETITION)
TASK_PATH=ROOT/f'{TASK_ID}.json'

OUT_DIR=Path.cwd()
ONNX_PATH=OUT_DIR/f'{TASK_ID}_static_graph.onnx'
ZIP_PATH=OUT_DIR/f'{TASK_ID}_static_graph_submission.zip'
GENERIC_ZIP=OUT_DIR/'submission.zip'


In [5]:
class Task074StaticSymmetry(nn.Module):
    """Static neural-symbolic repair for task074.

    Semantic rule:
    - color 9 is an occluder, not an output color;
    - the 28x28 core at rows/cols 2..29 has D4 symmetry;
    - the two top border rows and left border columns are mirrored strips;
    - non-occluded cells are preserved.
    """
    def __init__(self):
        super().__init__()
        self.register_buffer('z_top_left', torch.zeros(1,9,2,2))
        self.register_buffer('z_ch9', torch.zeros(1,1,30,30))
    def forward(self, x):
        non9 = x[:, :9, :, :]
        red = x[:, 9:10, :, :]

        core = non9[:, :, 2:30, 2:30]
        tr = core.transpose(2,3)
        core_sum = (core + torch.flip(core, [2]) + torch.flip(core, [3]) + torch.flip(core, [2,3])
                    + tr + torch.flip(tr, [2]) + torch.flip(tr, [3]) + torch.flip(tr, [2,3]))
        core_fill = (core_sum > 0.5).float()

        row_strip = non9[:, :, 0:2, 2:30]
        row_fill = ((row_strip + torch.flip(row_strip, [3])) > 0.5).float()

        col_strip = non9[:, :, 2:30, 0:2]
        col_fill = ((col_strip + torch.flip(col_strip, [2])) > 0.5).float()

        top = torch.cat([self.z_top_left, row_fill], dim=3)
        bottom = torch.cat([col_fill, core_fill], dim=3)
        fill = torch.cat([top, bottom], dim=2)
        fill = ((fill + non9.transpose(2,3)) > 0.5).float()

        out9 = non9 * (1.0 - red) + fill * red
        return torch.cat([out9, self.z_ch9], dim=1)

def ensure_tools():
    import onnx, onnxruntime as ort
    return onnx, ort

def grid_to_tensor(grid):
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    a=np.array(grid,dtype=np.int64)
    for r in range(a.shape[0]):
        for c in range(a.shape[1]):
            v=int(a[r,c])
            if 0<=v<CH:
                x[0,v,r,c]=1.0
    return x

def tensor_to_grid(y):
    y=np.asarray(y)
    if y.ndim==4: y=y[0]
    # Competition-style: one-hot/threshold. If multiple, argmax; here exact one channel per cell.
    return y.argmax(axis=0).astype(np.int64).tolist()

def exact_eval(sess, examples):
    exact=0; first=None
    inp=sess.get_inputs()[0].name
    for i,ex in enumerate(examples):
        y=sess.run(None,{inp:grid_to_tensor(ex['input'])})[0]
        pred=np.array(tensor_to_grid(y),dtype=np.int64)
        out=np.array(ex['output'],dtype=np.int64)
        ok=np.array_equal(pred,out)
        exact += int(ok)
        if not ok and first is None:
            first={'idx':i,'wrong_pixels':int((pred!=out).sum())}
    return {'exact':exact,'total':len(examples),'first_wrong':first}

def op_counts(model):
    return dict(collections.Counter(n.op_type for n in model.graph.node))

def onnx_shape(value_info):
    dims=[]
    for d in value_info.type.tensor_type.shape.dim:
        if d.dim_value: dims.append(int(d.dim_value))
        else: dims.append(None)
    return dims

def find_9_components(grid):
    a=np.array(grid,dtype=np.int64); seen=np.zeros_like(a,dtype=bool); comps=[]
    h,w=a.shape
    for r in range(h):
        for c in range(w):
            if a[r,c]==9 and not seen[r,c]:
                stack=[(r,c)]; seen[r,c]=True; pts=[]
                while stack:
                    rr,cc=stack.pop(); pts.append((rr,cc))
                    for dr,dc in [(1,0),(-1,0),(0,1),(0,-1)]:
                        nr,nc=rr+dr,cc+dc
                        if 0<=nr<h and 0<=nc<w and a[nr,nc]==9 and not seen[nr,nc]:
                            seen[nr,nc]=True; stack.append((nr,nc))
                rs=[p[0] for p in pts]; cs=[p[1] for p in pts]
                comps.append((min(rs),max(rs),min(cs),max(cs),len(pts)))
    return comps

def example_features(split, idx, ex, ok):
    a=np.array(ex['input'],dtype=np.int64); o=np.array(ex['output'],dtype=np.int64)
    comps=find_9_components(a)
    shapes=tuple(sorted((r1-r0+1,c1-c0+1) for r0,r1,c0,c1,n in comps))
    bboxes=tuple(sorted((r0,r1,c0,c1) for r0,r1,c0,c1,n in comps))
    redmask=a==9
    core_red=int(redmask[2:30,2:30].sum())
    rowstrip_red=int(redmask[0:2,2:30].sum())
    colstrip_red=int(redmask[2:30,0:2].sum())
    both_t_red=int(np.logical_and(redmask, redmask.T).sum())
    return {
        'split':split,'idx':idx,'exact':int(ok),
        'grid_size':str(tuple(a.shape)),
        'occluder_component_count':len(comps),
        'occluder_shapes':str(shapes),
        'occluder_bboxes':str(bboxes),
        'red_count':int(redmask.sum()),
        'core_red_count':core_red,
        'rowstrip_red_count':rowstrip_red,
        'colstrip_red_count':colstrip_red,
        'both_transpose_red_count':both_t_red,
        'output_color_set':str(tuple(sorted(map(int,set(o.ravel()))))),
    }

def write_structural_reports(task, sess, health):
    import pandas as pd
    REPORT_DIR.mkdir(parents=True,exist_ok=True)
    inp=sess.get_inputs()[0].name
    rows=[]
    for split in ['train','test','arc-gen']:
        for idx,ex in enumerate(task.get(split,[])):
            y=sess.run(None,{inp:grid_to_tensor(ex['input'])})[0]
            pred=np.array(tensor_to_grid(y),dtype=np.int64)
            ok=np.array_equal(pred,np.array(ex['output'],dtype=np.int64))
            rows.append(example_features(split,idx,ex,ok))
    df=pd.DataFrame(rows)
    df.to_csv(REPORT_DIR/'per_example_structural_features_and_accuracy.csv',index=False)
    group_rows=[]
    feature_cols=['grid_size','occluder_component_count','occluder_shapes','occluder_bboxes','red_count','core_red_count','rowstrip_red_count','colstrip_red_count','both_transpose_red_count','output_color_set']
    for feat in feature_cols:
        for val,sub in df.groupby(feat, dropna=False):
            group_rows.append({'feature':feat,'value':val,'n':len(sub),'exact':int(sub['exact'].sum()),'accuracy':float(sub['exact'].mean())})
    pd.DataFrame(group_rows).to_csv(REPORT_DIR/'structural_group_accuracy.csv',index=False)
    train_df=df[df['split']=='train']
    ag=df[df['split']=='arc-gen']
    hold=[]
    for feat in feature_cols:
        seen=set(train_df[feat].astype(str))
        mask=~ag[feat].astype(str).isin(seen)
        sub=ag[mask]
        hold.append({'feature':feat,'train_values':len(seen),'arc_values':ag[feat].nunique(),'heldout_examples':len(sub),'heldout_exact':int(sub['exact'].sum()),'accuracy':float(sub['exact'].mean()) if len(sub) else None})
    hold_df=pd.DataFrame(hold)
    hold_df.to_csv(REPORT_DIR/'structural_holdout_summary_direct_features.csv',index=False)
    with open(REPORT_DIR/'onnx_health.json','w') as f: json.dump(health,f,indent=2)
    md=[]
    md.append('# task074 static-graph structural validation\n')
    md.append('## ONNX health\n')
    md.append(f"- input shape: {health['input_shape']}\n")
    md.append(f"- output shape: {health['output_shape']}\n")
    md.append(f"- size bytes: {health['size_bytes']}\n")
    md.append(f"- forbidden ops: {health['forbidden_ops_present']}\n")
    md.append(f"- dynamic/scatter risk ops: {health['risk_ops_present']}\n")
    md.append(f"- op counts: {health['op_counts']}\n")
    md.append('\n## Accuracy\n')
    for key in ['train','test','arc-gen','arc_gen_70_30']:
        md.append(f"- {key}: {health['accuracy'][key]}\n")
    md.append('\n## Structural holdouts relative to visible train\n')
    md.append('| feature | heldout exact | accuracy |\n|---|---:|---:|\n')
    for _,r in hold_df.iterrows():
        if int(r['heldout_examples'])>0:
            md.append(f"| {r['feature']} | {int(r['heldout_exact'])}/{int(r['heldout_examples'])} | {r['accuracy']:.3f} |\n")
    md.append('\n## Interpretation\n')
    md.append('The model treats color 9 as an occlusion mask. It restores the 28x28 offset core using D4 symmetry and restores the first two border rows/columns using the mirrored strip rule, while preserving all non-occluded cells. The exported graph is fixed-shape and contains no dynamic-shape or scatter-style operators.\n')
    (REPORT_DIR/'task074_structural_validation_report.md').write_text(''.join(md))
    # zip
    if REPORT_ZIP.exists(): REPORT_ZIP.unlink()
    shutil.make_archive(str(REPORT_ZIP).replace('.zip',''), 'zip', REPORT_DIR)


In [6]:
onnx, ort = ensure_tools()
with open(TASK_PATH) as f: task=json.load(f)

In [7]:

model=Task074StaticSymmetry().eval()
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)

torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],
                  output_names=['output'],opset_version=13,
                  dynamic_axes=None,do_constant_folding=True,dynamo=False)

m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m)


ops=op_counts(m)
forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
risk={'ScatterND','Shape','Range','Expand','Gather','ConstantOfShape'}
health={
    'input_shape': onnx_shape(m.graph.input[0]),
    'output_shape': onnx_shape(m.graph.output[0]),
    'size_bytes': ONNX_PATH.stat().st_size,
    'op_counts': ops,
    'forbidden_ops_present': sorted(forbidden.intersection(ops)),
    'risk_ops_present': sorted(risk.intersection(ops)),
    'accuracy': {}
}


/tmp/ipykernel_16/3123711533.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],


In [8]:
sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
for split in ['train','test','arc-gen']:
    res=exact_eval(sess, task.get(split,[]))
    health['accuracy'][split]=f"{res['exact']}/{res['total']}"
    if res['first_wrong'] is not None:
        raise AssertionError((split,res))
ag=task.get('arc-gen',[])
fit=exact_eval(sess,ag[:183]); hold=exact_eval(sess,ag[183:])
health['accuracy']['arc_gen_70_30']=f"fit {fit['exact']}/{fit['total']}, test {hold['exact']}/{hold['total']}"
assert health['input_shape']==[1,10,30,30]
assert health['output_shape']==[1,10,30,30]
assert health['size_bytes']<1_400_000
assert not health['forbidden_ops_present']
assert not health['risk_ops_present']


In [9]:
ZIP_PATH

PosixPath('/kaggle/working/task074_static_graph_submission.zip')

In [10]:
# package zip with root task074.onnx
# GENERIC_ZIP is the '/kaggle/working/submission.zip'
for zp in [ZIP_PATH, GENERIC_ZIP]:
    if zp.exists(): zp.unlink()
    with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
